# Keeping people cool during a power outage

Compare standard response, earlier device outreach, and earlier outreach with powered cooling places. The interruption is already observed at hour zero. Utility inspections and completed work determine when power returns.

In [ ]:
from pathlib import Path

import napkin

here = Path.cwd()
entry = next(p for p in (here / 'model.scn', here / 'scenarios/heat_wave_blackout/model.scn') if p.is_file())
model = napkin.load_scenario(entry)
records = napkin.study(model)
assert all(r.failure is None for r in records.values())

## Service restoration and access to cooling

A service share of 0.5 means half the power supply has returned. Admissions count people who reached a powered cooling site.

In [ ]:
times = (0, 4, 12, 18, 25, 30, 35, 40, 60)
series = {}
for name, record in records.items():
    service, admissions = napkin.read(
        record, model.program, ['supply.share_restored', 'places.admissions'], times
    )
    series[name] = (service.values, admissions.values)
    print(name)
    print('Hour    Supply restored    People admitted to cooling')
    for hour, share, people in zip(times, service.values, admissions.values):
        print(f'{hour:>4}    {share:>14.1%}    {people:>26,.2f}')

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print('The table above gives the full series. Install Matplotlib to plot it here.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for name, (service, admissions) in series.items():
        axes[0].plot(times, service, label=name)
        axes[1].plot(times, admissions, label=name)
    for ax, label in zip(axes, ('Supply share restored', 'People admitted to powered cooling sites')):
        ax.set(xlabel='Hours since the outage', ylabel=label)
    axes[1].legend()
    fig.tight_layout()

## Device outreach

Read the number of people admitted after their home device loses power at the end of each response. Each reading is in people.

In [ ]:
for name, record in records.items():
    (admitted,) = napkin.read(record, model.program, ['device_admissions'], [record.end])
    print(f'{name}: {admitted.values[0]:.2f} people')